# General Health Query Chatbot (Prompt Engineering + OpenRouter)

A Kaggle-ready notebook for a safe, prompt-engineered health assistant.

## 1. Problem Statement
Build a chatbot that answers general health questions using an LLM, while blocking unsafe or irrelevant prompts and keeping responses clear, friendly, and cautious.

## 2. Imports

In [14]:
import os
import re
import textwrap
from typing import List, Dict, Tuple

import requests
import pandas as pd

print("Imports ready.")

Imports ready.


In [15]:
import os
from kaggle_secrets import UserSecretsClient

# -------------------------------------------------
# Load OpenRouter API key from Kaggle Secrets
# -------------------------------------------------
user_secrets = UserSecretsClient()
os.environ["OPENROUTER_API_KEY"] = user_secrets.get_secret("OPENROUTER_API_KEY")

# -------------------------------------------------
# OpenRouter configuration
# -------------------------------------------------
OPENROUTER_API_URL = "https://openrouter.ai/api/v1/chat/completions"

# Better than openrouter/free for reproducibility
MODEL_ID = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-120b:free")
FALLBACK_MODEL_ID = os.getenv("OPENROUTER_FALLBACK_MODEL", "openai/gpt-oss-20b:free")

APP_TITLE = os.getenv("OPENROUTER_APP_TITLE", "Health Query Chatbot Notebook")
APP_REFERER = os.getenv("OPENROUTER_HTTP_REFERER", "https://kaggle.com")

API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip()

print("Model:", MODEL_ID)
print("Fallback model:", FALLBACK_MODEL_ID)
print("API key found:", bool(API_KEY))

Model: openai/gpt-oss-120b:free
Fallback model: openai/gpt-oss-20b:free
API key found: True


## 3. OpenRouter Configuration
The notebook uses OpenRouter's chat-completions API with Bearer authentication. The default model is the free router, which can be swapped for any `:free` model.

## 4. Dynamic System Prompt
This prompt makes the assistant helpful, safe, and concise.

In [17]:
def build_system_prompt() -> str:
    return textwrap.dedent('''
    You are MedGuide, a helpful and calm medical information assistant.

    Behavior:
    - Answer general health-related questions in plain language.
    - Be friendly, clear, and concise.
    - Ask brief follow-up questions when needed.
    - Never pretend to diagnose or prescribe.
    - Never give harmful or reckless medical advice.
    - For emergencies (chest pain, difficulty breathing, stroke symptoms, severe bleeding, fainting, seizure), tell the user to seek urgent medical help immediately.
    - For self-harm, overdose, or violence, provide a supportive emergency-style response.
    - For medicines, children, pregnancy, or complex conditions, add a caution to consult a clinician or pharmacist.
    - If the user asks a non-health question, politely refuse and redirect to a health question.
    - Keep most responses under 250 words unless detailed explanation is required.

    Response style:
    - Use simple language.
    - Use bullets when helpful.
    - Include "When to seek help" if appropriate.
    - Keep the response safe and responsible.
    ''').strip()

print(build_system_prompt()[:400], "...")

You are MedGuide, a helpful and calm medical information assistant.

Behavior:
- Answer general health-related questions in plain language.
- Be friendly, clear, and concise.
- Ask brief follow-up questions when needed.
- Never pretend to diagnose or prescribe.
- Never give harmful or reckless medical advice.
- For emergencies (chest pain, difficulty breathing, stroke symptoms, severe bleeding, fa ...


## 5. Safety Filters and Query Validation

In [18]:
EMERGENCY_PATTERNS = [
    r"chest pain", r"difficulty breathing", r"shortness of breath", r"stroke",
    r"face droop", r"one-sided weakness", r"severe bleeding", r"unconscious",
    r"fainting", r"seizure", r"heart attack", r"anaphylaxis"
]

UNSAFE_PATTERNS = [
    r"suicide", r"kill myself", r"self harm", r"self-harm", r"overdose",
    r"take too much", r"hurt myself", r"hurt others"
]

HEALTH_KEYWORDS = [
    "symptom", "symptoms", "pain", "fever", "cough", "throat", "headache",
    "nausea", "vomiting", "dizziness", "rash", "infection", "cold", "flu",
    "allergy", "asthma", "diabetes", "blood pressure", "cholesterol", "heart",
    "kidney", "liver", "lung", "stomach", "diarrhea", "constipation",
    "medication", "medicine", "drug", "tablet", "dose", "dosage", "paracetamol",
    "acetaminophen", "ibuprofen", "antibiotic", "doctor", "clinic", "hospital",
    "diagnosis", "treatment", "pregnancy", "pregnant", "child", "children",
    "baby", "infant", "mental health", "anxiety", "depression", "panic"
]

NON_HEALTH_HINTS = [
    "python", "java", "javascript", "sql", "programming", "coding", "algorithm",
    "math", "write a poem", "story", "essay", "translate", "finance", "stock",
    "sports", "news", "movie", "song", "recipe"
]

def _normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower().strip())

def detect_emergency(query: str) -> bool:
    q = _normalize(query)
    return any(re.search(p, q) for p in EMERGENCY_PATTERNS)

def detect_unsafe(query: str) -> bool:
    q = _normalize(query)
    return any(re.search(p, q) for p in UNSAFE_PATTERNS)

def is_health_related(query: str) -> bool:
    q = _normalize(query)
    if any(hint in q for hint in NON_HEALTH_HINTS) and not any(k in q for k in HEALTH_KEYWORDS):
        return False
    if any(k in q for k in HEALTH_KEYWORDS):
        return True
    fallback_patterns = [
        r"what causes", r"why do i", r"what should i do for", r"is .* safe",
        r"could this be", r"does .* mean", r"what are the symptoms",
        r"how to treat", r"how do i treat"
    ]
    return any(re.search(p, q) for p in fallback_patterns)

def gate_query(query: str) -> Tuple[bool, str]:
    q = _normalize(query)

    prompt_injection_patterns = [
        "ignore previous instructions",
        "act as",
        "system prompt",
        "developer mode",
        "bypass",
        "jailbreak",
    ]

    if any(p in q for p in prompt_injection_patterns):
        return False, "This assistant only supports safe general health-related questions."

    if detect_unsafe(query):
        return False, "This looks like a self-harm or harm-related message. Please contact local emergency services or a trusted person right now."

    if detect_emergency(query):
        return False, "This may be an emergency. Seek urgent medical help now or call your local emergency number."

    if not is_health_related(query):
        return False, "I can only help with health-related questions. Please ask a general health, symptom, medicine, or wellness question."

    return True, "health"

print("Safety filters ready.")

Safety filters ready.


## 6. Conversation Memory

In [19]:
conversation_history: List[Dict[str, str]] = []

def trim_history(history: List[Dict[str, str]], max_messages: int = 8) -> List[Dict[str, str]]:
    return history[-max_messages:]

def build_messages(user_query: str, history: List[Dict[str, str]]) -> List[Dict[str, str]]:
    messages = [{"role": "system", "content": build_system_prompt()}]
    messages.extend(trim_history(history))
    messages.append({"role": "user", "content": user_query})
    return messages

## 7. OpenRouter API Call

In [20]:
def call_openrouter(messages: List[Dict[str, str]],
                    model: str = MODEL_ID,
                    fallback_model: str = FALLBACK_MODEL_ID,
                    temperature: float = 0.2,
                    max_tokens: int = 350,
                    top_p: float = 0.9,
                    timeout: int = 60) -> str:
    if not API_KEY:
        raise ValueError(
            "OPENROUTER_API_KEY was not found. Add it in Kaggle Secrets."
        )

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": APP_REFERER,
        "X-OpenRouter-Title": APP_TITLE,
    }

    def _send_request(selected_model: str):
        payload = {
            "model": selected_model,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
            "top_p": top_p,
        }
        resp = requests.post(
            OPENROUTER_API_URL,
            headers=headers,
            json=payload,
            timeout=timeout
        )
        return resp

    # Try primary model
    resp = _send_request(model)

    # Fallback if needed
    if resp.status_code != 200 and fallback_model and fallback_model != model:
        resp = _send_request(fallback_model)

    if resp.status_code == 429:
        return "The service is currently busy. Please try again shortly."
    
    if resp.status_code != 200:
        raise RuntimeError(f"OpenRouter API error {resp.status_code}: {resp.text[:500]}")


    data = resp.json()
    content = data["choices"][0]["message"].get("content")

    if content is None or not str(content).strip():
        return (
            "I could not generate a reliable health response at the moment. "
            "Please try rephrasing your question."
        )
    
    return str(content).strip()

## 8. Main Chat Function

In [21]:
def health_chat(user_query: str, history: List[Dict[str, str]] = None, model: str = MODEL_ID) -> str:
    if history is None:
        history = []

    allowed, reason = gate_query(user_query)
    if not allowed:
        return reason

    messages = build_messages(user_query, history)
    answer = call_openrouter(messages=messages, model=model)

    history.append({"role": "user", "content": user_query})
    history.append({"role": "assistant", "content": answer})
    return answer

print("Chatbot function ready.")

Chatbot function ready.


## 9. Example Queries / Validation

In [22]:
demo_queries = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?",
    "I have chest pain and difficulty breathing",
    "Write a poem about the moon"
]

demo_history: List[Dict[str, str]] = []
rows = []

for q in demo_queries:
    try:
        ans = health_chat(q, demo_history)
    except Exception as e:
        ans = f"[ERROR] {e}"
    rows.append({"Query": q, "Response": ans})

pd.DataFrame(rows)

,Query,Response
0,What causes a sore throat?,**Commoncauses of a sore throat**\n\n- **Viral...
1,Is paracetamol safe for children?,I could not generate a reliable health respons...
2,I have chest pain and difficulty breathing,This may be an emergency. Seek urgent medical ...
3,Write a poem about the moon,I can only help with health-related questions....


## 10. Optional Interactive Chat

In [23]:
def run_interactive_chat():
    local_history: List[Dict[str, str]] = []
    print("Health chatbot ready. Type 'exit' to stop.\n")
    while True:
        q = input("You: ").strip()
        if q.lower() in {"exit", "quit"}:
            print("Chat ended.")
            break
        try:
            ans = health_chat(q, local_history)
        except Exception as e:
            ans = f"[ERROR] {e}"
        print(f"Assistant: {ans}\n")

# Uncomment to use:
run_interactive_chat()

Health chatbot ready. Type 'exit' to stop.



You:  hi, how are u?


Assistant: I can only help with health-related questions. Please ask a general health, symptom, medicine, or wellness question.



You:  Why do I feel dizzy when standing up quickly?


Assistant: Feeling a brief “light‑headed” or dizzy sensation when you stand up quickly is usually caused by a temporary drop in blood pressure as gravity pulls blood down into your legs. This is called **orthostatic (or postural) hypotension**.  

### Common reasons
- **Dehydration** – not drinking enough fluids or losing fluid from sweating, illness, or diuretics.
- **Low blood volume** – from bleeding, heavy menstrual periods, or certain medications.
- **Medications** – especially blood‑pressure drugs, diuretics, antidepressants, or drugs that relax blood vessels.
- **Prolonged sitting or bed rest** – the body’s reflexes that normally keep blood pressure steady become a bit slower.
- **Alcohol** or large meals can also lower blood pressure temporarily.
- **Underlying health conditions** such as heart problems, endocrine disorders (e.g., adrenal insufficiency), or nervous‑system diseases.

### What you can try
- **Rise slowly**: Sit on the edge of the bed or chair for a minute before 

...

Assistant: I can only help with health-related questions. Please ask a general health, symptom, medicine, or wellness question.



You:  Do I definitely have cancer if I lose weight suddenly?


Assistant: I can only help with health-related questions. Please ask a general health, symptom, medicine, or wellness question.



You:  exit


Chat ended.


## 11. Final Insights
This project combines prompt engineering, LLM API integration, safety guardrails, and response control for a health assistant. The strict validator blocks irrelevant questions, while the system prompt keeps the model helpful and cautious.